In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [2]:
###Data Preparation:
###Loading the dataset:
df=pd.read_csv('anime.csv')

In [3]:
df

,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266
...,...,...,...,...,...,...,...
12289,9316,Toushindai My Lover: Minami tai Mecha-Minami,Hentai,OVA,1,4.15,211
12290,5543,Under World,Hentai,OVA,1,4.28,183
12291,5621,Violence Gekiga David no Hoshi,Hentai,OVA,4,4.88,219
12292,6133,Violence Gekiga Shin David no Hoshi: Inma Dens...,Hentai,OVA,1,4.98,175


In [4]:
df.head()

,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266


In [5]:
df.shape

(12294, 7)

In [6]:
df.size

86058

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12294 entries, 0 to 12293
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   anime_id  12294 non-null  int64  
 1   name      12294 non-null  object 
 2   genre     12232 non-null  object 
 3   type      12269 non-null  object 
 4   episodes  12294 non-null  object 
 5   rating    12064 non-null  float64
 6   members   12294 non-null  int64  
dtypes: float64(1), int64(2), object(4)
memory usage: 672.5+ KB


In [8]:
###Checking Missing Values:
df.isnull().sum()

anime_id      0
name          0
genre        62
type         25
episodes      0
rating      230
members       0
dtype: int64

In [9]:
###Handling Missing Values:
df = df.dropna()

In [10]:
df.isnull().sum()

anime_id    0
name        0
genre       0
type        0
episodes    0
rating      0
members     0
dtype: int64

In [11]:
####Exploring the dataset to understand its structure and attributes:
df.columns

Index(['anime_id', 'name', 'genre', 'type', 'episodes', 'rating', 'members'], dtype='object')

In [12]:
df.describe()

,anime_id,rating,members
count,12017.000000,12017.000000,1.201700e+04
mean,13638.001165,6.478264,1.834888e+04
std,11231.076675,1.023857,5.537250e+04
min,1.000000,1.670000,1.200000e+01
25%,3391.000000,5.890000,2.250000e+02
50%,9959.000000,6.570000,1.552000e+03
75%,23729.000000,7.180000,9.588000e+03
max,34519.000000,10.000000,1.013917e+06


In [13]:
df.dtypes

anime_id      int64
name         object
genre        object
type         object
episodes     object
rating      float64
members       int64
dtype: object

## Explanation:
        The dataset contains both categorical and numerical attributes such as anime name, genre, type, episodes, rating, and members, which are useful for recommendation analysis.

In [14]:
####Feature Extraction:
# Features selected for similarity
features = df[['genre', 'rating']]
features.head()

,genre,rating
0,"Drama, Romance, School, Supernatural",9.37
1,"Action, Adventure, Drama, Fantasy, Magic, Mili...",9.26
2,"Action, Comedy, Historical, Parody, Samurai, S...",9.25
3,"Sci-Fi, Thriller",9.17
4,"Action, Comedy, Historical, Parody, Samurai, S...",9.16


The features chosen for similarity computation are genre and rating. Genre captures content similarity, while rating reflects user preference and quality.

In [15]:
# Convert genre text into numerical vectors
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(stop_words='english')
genre_matrix = tfidf.fit_transform(df['genre'].astype(str))
genre_matrix.shape

(12017, 46)

The categorical feature genre was converted into numerical vectors using TF-IDF vectorization so that cosine similarity could be computed.

In [16]:
# Normalize rating column:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
df[['rating']] = scaler.fit_transform(df[['rating']])
df[['rating']].head()

,rating
0,0.924370
1,0.911164
2,0.909964
3,0.900360
4,0.899160


The numerical feature rating was normalized using Min-Max Scaling so that all values lie between 0 and 1 before similarity computation.

In [17]:
###Recommendation System:
###Design a function to recommend anime based on cosine similarity:
from sklearn.metrics.pairwise import cosine_similarity
cosine_sim = cosine_similarity(genre_matrix)
# Create index using anime names
indices = pd.Series(df.index, index=df['name']).drop_duplicates()
# Recommendation function
def recommend_anime(title, top_n=5):
    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n+1]

    anime_indices = [i[0] for i in sim_scores]
    return df['name'].iloc[anime_indices]

This function takes an anime title, calculates cosine similarity with all anime, sorts them by similarity score, and returns the top recommended anime.

In [18]:
#####Given a target anime, recommend a list of similar anime based on cosine similarity scores:
# Example: recommend anime similar to Naruto:
recommendations = recommend_anime('Naruto', top_n=5)
print(recommendations)

615                                    Naruto: Shippuuden
841                                                Naruto
1103    Boruto: Naruto the Movie - Naruto ga Hokage ni...
1343                                          Naruto x UT
1472          Naruto: Shippuuden Movie 4 - The Lost Tower
Name: name, dtype: object


For the target anime Naruto, the system recommends the top 5 anime with the highest cosine similarity scores, indicating that they are most similar in genre/content.

In [19]:
####Experiment with different threshold values for similarity scores:
def recommend_threshold(title, threshold=0.3):
    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    filtered = [x for x in sim_scores if x[1] > threshold and x[0] != idx]
    anime_indices = [i[0] for i in filtered]
    return df['name'].iloc[anime_indices]
# Different threshold values
print(recommend_threshold('Naruto', threshold=0.2))
print(recommend_threshold('Naruto', threshold=0.5))

6                                   Hunter x Hunter (2011)
13                      Code Geass: Hangyaku no Lelouch R2
19                         Code Geass: Hangyaku no Lelouch
20                                          Hajime no Ippo
21       Rurouni Kenshin: Meiji Kenkaku Romantan - Tsui...
                               ...                        
11930                                V.G.Neo The Animation
12020                                 Sexy Sailor Soldiers
12106                             Sailor Senshi Venus♥Five
12146                                           Dochinpira
12221                           Kunoichi Gakuen Ninpouchou
Name: name, Length: 1458, dtype: object
6            Hunter x Hunter (2011)
74                        One Piece
86               Shingeki no Kyojin
106                    Katanagatari
112                 Hunter x Hunter
                    ...            
11528                    Kage (OVA)
11565        Taimanin Asagi Special
11930         V.G.Neo T

Increasing the similarity threshold reduces the recommendation list size and improves recommendation relevance.

## Analyzing the performance of the recommendation system :
        *The recommendation system provides anime with similar genres using cosine similarity.

        *Recommendations are relevant when the target anime has clear genre information.

        *Higher similarity thresholds give more accurate recommendations, while lower thresholds give more results.

        *The system uses only content features, so it may miss user-specific preferences.

## Identifying areas of improvement:
        *Include additional features such as episodes, members, and ratings.

        *Use user rating history for personalized recommendations.

        *Combine content-based and collaborative filtering methods.

        *Improve preprocessing of genre text for better similarity calculation.

## Conclusion: 
            The cosine similarity–based recommendation system works well for genre-based recommendations, but adding more features and user behavior data can improve accuracy and personalization.

## Interview Questions:
## 1. Can you explain the difference between user-based and item-based collaborative filtering?
 ## User-based collaborative filtering: 
     It recommends items liked by users with similar preferences.
 ## Item-based collaborative filtering:
     It recommends items that are similar to the item already liked by the user.

## 2. What is collaborative filtering, and how does it work?

         Collaborative filtering is a recommendation technique that predicts user interests by analyzing the preferences or ratings of many users. It works by finding similar users or similar items and recommending items based on those similarities.